## 1. Imports
Import the necessary PyTorch modules and pretrained model weights.

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

## 2. Model Definition
Define the `MultiLevelCBM` class with a ResNet-50 backbone and three output heads.

In [ ]:
class MultiLevelCBM(nn.Module):
    """
    Multi-Level Concept Bottleneck Model.

    Architecture:
        Backbone  : ResNet-50 (pretrained on ImageNet) — outputs a 2048-d feature vector.
        L1 head   : Linear(2048 → num_l1)   — coarse concept logits   (7 groups)
        L2 head   : Linear(2048 → num_l2)   — fine attribute logits   (312 attrs)
        Classifier: Linear(2048 → num_cls)  — class logits            (200 species)

    All three heads share the same backbone features (hard parameter sharing).
    """

    def __init__(self, num_classes: int = 200, num_l1: int = 7, num_l2: int = 312):
        super().__init__()

        # ── Backbone ──────────────────────────────────────────────────────────
        # Load ResNet-50 with ImageNet weights, then strip the final FC layer.
        # What remains outputs (B, 2048, 1, 1) after global average pooling.
        backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        self.features = nn.Sequential(*list(backbone.children())[:-1])

        # ── Heads ─────────────────────────────────────────────────────────────
        self.l1_head     = nn.Linear(2048, num_l1)      # coarse concepts
        self.l2_head     = nn.Linear(2048, num_l2)      # fine attributes
        self.classifier  = nn.Linear(2048, num_classes) # final class

    def forward(self, x: torch.Tensor):
        """
        Args:
            x: (B, 3, 224, 224)  — batch of RGB images

        Returns:
            cls_logits : (B, 200)   — class logits
            l1_logits  : (B, 7)    — coarse concept logits
            l2_logits  : (B, 312)  — fine attribute logits
        """
        feats = self.features(x)    # (B, 2048, 1, 1)
        feats = feats.flatten(1)    # (B, 2048)

        cls_logits = self.classifier(feats)  # (B, 200)
        l1_logits  = self.l1_head(feats)     # (B, 7)
        l2_logits  = self.l2_head(feats)     # (B, 312)

        return cls_logits, l1_logits, l2_logits


print("MultiLevelCBM class defined.")

## 3. Instantiate and Inspect the Model
Create a model instance and print the total number of trainable parameters.

In [ ]:
model = MultiLevelCBM(num_classes=200, num_l1=7, num_l2=312)

# Count trainable parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")
print()
print("── Head shapes ─────────────────────────")
print(f"  Backbone output  : 2048")
print(f"  L1 head          : 2048 → {model.l1_head.out_features}")
print(f"  L2 head          : 2048 → {model.l2_head.out_features}")
print(f"  Classifier head  : 2048 → {model.classifier.out_features}")

## 4. Verify Forward Pass
Send a dummy batch through the model to confirm output shapes are correct before connecting the real DataLoader.

In [ ]:
model.eval()

dummy_input = torch.randn(4, 3, 224, 224)   # batch of 4 fake images

with torch.no_grad():
    cls_out, l1_out, l2_out = model(dummy_input)

print("Forward pass successful!")
print(f"  cls_logits shape : {tuple(cls_out.shape)}  ← (batch, 200 classes)")
print(f"  l1_logits  shape : {tuple(l1_out.shape)}    ← (batch, 7 coarse concepts)")
print(f"  l2_logits  shape : {tuple(l2_out.shape)}  ← (batch, 312 fine attributes)")

## 5. Select Device
Move the model to GPU if available, otherwise fall back to CPU.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = model.to(device)

print(f"Device : {device}")
if device.type == 'cuda':
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

## Model Architecture Roadmap

**Phase 2 of 5** in the XAI project pipeline.

The `MultiLevelCBM` model takes a bird image as input and simultaneously predicts three outputs using a shared ResNet-50 backbone.

```
Input Image (3, 224, 224)
        │
        ▼
  Backbone (ResNet-50, pretrained on ImageNet)
        │
   features (2048,)
        │
   ┌────┼──────────┐
   ▼    ▼          ▼
 L1    L2      Classifier
(→7) (→312)     (→200)
```

### 1. Imports
**Output:** `torch`, `nn`, `resnet50`, `ResNet50_Weights` ready for use.

### 2. Model Definition
**Input:** Nothing — defines the class.

**Process:** Loads ResNet-50 with ImageNet weights, strips the final FC layer, and attaches three linear heads sharing the same 2048-d feature vector.
**Output:** `MultiLevelCBM` class with `.features`, `.l1_head`, `.l2_head`, `.classifier`.

### 3. Instantiate and Inspect
**Input:** `MultiLevelCBM` class.

**Process:** Creates the model and counts all trainable parameters.

**Output:** `model` — initialized instance (~25M parameters).

### 4. Verify Forward Pass
**Input:** `model` + a random dummy tensor of shape `(4, 3, 224, 224)`.

**Process:** Runs a single forward pass with `torch.no_grad()` to verify shapes without computing gradients.

**Output:**
```
cls_logits : (4, 200)   ← 200 bird species
l1_logits  : (4,   7)   ← 7 coarse concepts
l2_logits  : (4, 312)   ← 312 fine attributes
```

### 5. Select Device
**Input:** `model`.

**Process:** Detects CUDA availability and moves the model to the appropriate device.

**Output:** `device` — either `cuda` or `cpu`. `model` moved to that device.

### Full Pipeline Overview

| Component | Input shape | Output shape | Role |
|-----------|-------------|--------------|------|
| **Backbone** (ResNet-50) | (B, 3, 224, 224) | (B, 2048) | Extract visual features |
| **L1 Head** | (B, 2048) | (B, 7) | Coarse concept logits |
| **L2 Head** | (B, 2048) | (B, 312) | Fine attribute logits |
| **Classifier** | (B, 2048) | (B, 200) | Bird species logits |

**Next step → Phase 3: Training** (`train.ipynb`)
- Define multi-task loss: `L_total = L_cls + λ₁·L_l1 + λ₂·L_l2`
- Connect `train_loader` from `data.ipynb`
- Train and save checkpoints